<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/4.3%20Using%20Tools%20and%20Memory/4%20Tutorial%20-%20Adding%20Cache%20to%20a%20LangChain%20Project%20(InMemoryCache)%20Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Install
!pip install -q langchain-core==1.6.3 langchain-openai==1.6.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 4.7 MB/s eta 0:00:00


## Tutorial: Adding Cache to a LangChain Project with InMemoryCache
Speed up repeated queries with a simple, drop‑in cache.


In [2]:
import os, time
from getpass import getpass
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Enter your OpenRouter API key securely when prompted.
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")

# OpenRouter provides an OpenAI-compatible API.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# You can change this to any compatible OpenRouter model.
MODEL = "openai/gpt-4.1-mini"

llm = ChatOpenAI(
    model=MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    seed=42,
)

print("OpenRouter configured successfully.")
print("Model:", MODEL)

qa_prompt = PromptTemplate.from_template(
    "Answer briefly: {question}"
)
qa_chain = qa_prompt | llm | StrOutputParser()


Enter your OpenRouter API key: ··········
OpenRouter configured successfully.
Model: openai/gpt-4.1-mini


### Step 1: Run the chain twice without cache
We’ll measure elapsed time and compare after enabling cache.


In [4]:
q = "What is LangChain in one sentence?"

start = time.time()
print(qa_chain.invoke({"question": q}))
print(f"First run: {time.time() - start:.2f}s")

start = time.time()
print(qa_chain.invoke({"question": q}))
print(f"Second run (no cache): {time.time() - start:.2f}s")


LangChain is a framework for building applications that integrate large language models with external data and tools.
First run: 2.51s
LangChain is a framework for building applications that integrate large language models with external data and tools.
Second run (no cache): 1.09s


### Step 2: Enable InMemoryCache
We’ll set a global LLM cache and repeat the calls to see the speedup.


In [5]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache

set_llm_cache(InMemoryCache())

start = time.time()
print(qa_chain.invoke({"question": q}))
print(f"Third run (cached): {time.time() - start:.2f}s")

start = time.time()
print(qa_chain.invoke({"question": q}))
print(f"Fourth run (cached): {time.time() - start:.2f}s")


LangChain is a framework for building applications that integrate large language models with external data and tools.
Third run (cached): 1.46s
LangChain is a framework for building applications that integrate large language models with external data and tools.
Fourth run (cached): 0.00s


### Step 3: Notes on production caches
- Use Redis/SQLite/Postgres caches for persistence across processes and restarts
- Consider selective caching (e.g., only cache deterministic prompts)
- Invalidate on data or prompt changes to avoid stale answers